# Lesson 4b: Regularisation and Generalisation — Practical

4a derived bias-variance, weight decay, dropout, early stopping, data
augmentation and double descent from first principles and hand-built every
regulariser in NumPy. This notebook reproduces the practical side with
PyTorch's production building blocks — `torchvision.transforms`,
`nn.Dropout`, an optimiser's `weight_decay` argument — and runs three
experiments 4a's from-scratch tooling made awkward: a genuine train /
validation / test split, a real early-stopping loop with checkpointing
(not just reading a finished curve after the fact), and a joint
augmentation-pipeline comparison.

By the end of this notebook you will have:
- compared **four augmentation pipelines** of increasing strength and
  quantified their effect on held-out validation accuracy,
- run a **dropout-rate x weight-decay ablation** with full learning
  curves, using `torch.optim`'s built-in `weight_decay` argument under
  plain SGD — exactly the equivalence 4a proved,
- implemented **early stopping** as a real training-loop mechanism with
  patience and checkpoint restoration on a genuine validation split, and
- combined the winning configuration into one **fully regularised model**
  and compared its learning curves directly against an unregularised
  baseline.

## Introduction

4a's regularisation experiments compared train and test curves directly,
because a from-scratch NumPy loop makes a third split awkward to wire in.
Production practice never does this: **test data is touched exactly
once**, at the very end, and every decision made *during* development
(which hyperparameters, when to stop) is made against a **validation**
split instead. This notebook introduces that three-way split and uses it
throughout — augmentation pipelines are compared on validation accuracy,
the dropout/weight-decay grid is scored on validation loss, and early
stopping watches validation loss to decide when to restore the best
checkpoint. Test accuracy appears exactly once, at the end, on the single
configuration this notebook actually recommends.

## Setup

In [ ]:
# Fixed seeds: every stochastic step in this notebook (weight init, minibatch
# order, dropout masks, augmentation, data subsampling) is reproducible.
import copy
import io
import pathlib
import urllib.request

import numpy as np
import torch

SEED = 0
np.random.seed(SEED)
torch.manual_seed(SEED)

import pandas as pd
import torch.nn as nn
import torch.optim as optim
import torchvision.transforms as T
import matplotlib.pyplot as plt
from PIL import Image

plt.rcParams["figure.figsize"] = (5, 4)
print("numpy:", np.__version__)
print("pandas:", pd.__version__)
print("torch:", torch.__version__)

### Device check

This notebook is written to run unmodified in Google Colab or locally.

In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("using device:", device)

### Loading CIFAR-10 with a train / validation / test split

Same source as 4a: the Hugging Face `uoft-cs/cifar10` parquet mirror. A
validation split is carved out of the *training* pool by loading
`n_train + n_val` examples under one random permutation and slicing —
avoiding any chance of the same image landing in both splits, which two
independent random draws could not otherwise guarantee.

In [ ]:
CIFAR_BASE = "https://huggingface.co/datasets/uoft-cs/cifar10/resolve/main/plain_text"


def load_cifar10_subset(split, n, seed):
    path = pathlib.Path("data") / f"cifar10_{split}.parquet"
    path.parent.mkdir(exist_ok=True)
    if not path.exists():
        urllib.request.urlretrieve(f"{CIFAR_BASE}/{split}-00000-of-00001.parquet", path)
    df = pd.read_parquet(path)
    g = np.random.default_rng(seed)
    idx = g.permutation(len(df))[:n]
    images = np.stack([
        np.asarray(Image.open(io.BytesIO(df.iloc[i]["img"]["bytes"])), dtype=np.float32) / 255.0
        for i in idx
    ])  # (n, 32, 32, 3), pixel values in [0, 1]
    labels = df.iloc[idx]["label"].to_numpy().astype(np.int64)
    return images, labels


def load_train_val_split(n_train, n_val, seed):
    images, labels = load_cifar10_subset("train", n_train + n_val, seed)
    return images[:n_train], labels[:n_train], images[n_train:], labels[n_train:]


def to_chw(images_hwc):
    return torch.tensor(images_hwc).permute(0, 3, 1, 2).contiguous()  # (n, 3, 32, 32)


CLASS_NAMES = ["airplane", "automobile", "bird", "cat", "deer",
               "dog", "frog", "horse", "ship", "truck"]

N_TRAIN, N_VAL, N_TEST = 300, 150, 150
images_train, labels_train, images_val, labels_val = load_train_val_split(N_TRAIN, N_VAL, seed=SEED)
images_test, labels_test = load_cifar10_subset("test", N_TEST, seed=SEED + 1)

X_train_img = to_chw(images_train)                      # kept as images for augmentation
y_train = torch.tensor(labels_train)
X_val = to_chw(images_val).reshape(N_VAL, -1).to(device)
y_val = torch.tensor(labels_val).to(device)
X_test = to_chw(images_test).reshape(N_TEST, -1).to(device)
y_test = torch.tensor(labels_test).to(device)

print(f"train: {N_TRAIN}   val: {N_VAL}   test: {N_TEST}")

fig, axes = plt.subplots(1, 8, figsize=(13, 2))
for ax, i in zip(axes, range(8)):
    ax.imshow(images_train[i])
    ax.set_title(CLASS_NAMES[labels_train[i]], fontsize=9)
    ax.axis("off")
plt.suptitle("CIFAR-10 training samples")
plt.show()

In [ ]:
def build_model(dropout_prob=0.0, seed=SEED):
    torch.manual_seed(seed)
    layers = [nn.Linear(3072, 128), nn.ReLU()]
    if dropout_prob > 0.0:
        layers.append(nn.Dropout(dropout_prob))
    layers += [nn.Linear(128, 64), nn.ReLU()]
    if dropout_prob > 0.0:
        layers.append(nn.Dropout(dropout_prob))
    layers.append(nn.Linear(64, 10))
    model = nn.Sequential(*layers)
    for m in model:
        if isinstance(m, nn.Linear):
            nn.init.kaiming_normal_(m.weight, nonlinearity="relu")
            nn.init.zeros_(m.bias)
    return model


criterion = nn.CrossEntropyLoss()


def train_epoch(model, optimizer, transform, epoch_seed):
    model.train()
    n = X_train_img.shape[0]
    order = torch.randperm(n, generator=torch.Generator().manual_seed(epoch_seed))
    losses = []
    for start in range(0, n, 32):
        idx = order[start:start + 32]
        imgs = X_train_img[idx]
        if transform is not None:
            imgs = torch.stack([transform(img) for img in imgs])
        xb = imgs.reshape(imgs.shape[0], -1).to(device)
        yb = y_train[idx].to(device)
        optimizer.zero_grad()
        loss = criterion(model(xb), yb)
        loss.backward()
        optimizer.step()
        losses.append(loss.item())
    return float(np.mean(losses))


def evaluate(model, X, y):
    model.eval()
    with torch.no_grad():
        logits = model(X)
        loss = criterion(logits, y).item()
        acc = (logits.argmax(dim=1) == y).float().mean().item()
    return loss, acc

## Augmentation Pipelines

4a implemented one hand-written augmentation (crop + flip). Production
code composes several `torchvision.transforms` into a pipeline and treats
the *choice* of pipeline as a hyperparameter in its own right. Four
pipelines of increasing strength, trained under identical conditions
otherwise, isolate how much each additional transform actually buys:

In [ ]:
pipelines = {
    "none": None,
    "flip": T.RandomHorizontalFlip(),
    "flip+crop": T.Compose([
        T.RandomCrop(32, padding=4, padding_mode="reflect"),
        T.RandomHorizontalFlip(),
    ]),
    "flip+crop+jitter": T.Compose([
        T.RandomCrop(32, padding=4, padding_mode="reflect"),
        T.RandomHorizontalFlip(),
        T.ColorJitter(brightness=0.2, contrast=0.2),
    ]),
}

EPOCHS_AUG = 60
aug_rows = []
aug_val_curves = {}
for name, transform in pipelines.items():
    model = build_model().to(device)
    optimizer = optim.Adam(model.parameters(), lr=1e-3)
    val_losses = []
    for epoch in range(EPOCHS_AUG):
        train_epoch(model, optimizer, transform, epoch_seed=epoch)
        val_loss, val_acc = evaluate(model, X_val, y_val)
        val_losses.append(val_loss)
    aug_val_curves[name] = val_losses
    aug_rows.append({"pipeline": name, "val_loss": val_loss, "val_acc": val_acc})

aug_df = pd.DataFrame(aug_rows).sort_values("val_acc", ascending=False).reset_index(drop=True)
print(aug_df.to_string(index=False, float_format=lambda v: f"{v:.3f}"))

fig, ax = plt.subplots(figsize=(6, 4))
ax.bar(aug_df["pipeline"], aug_df["val_acc"])
ax.set_ylabel("validation accuracy")
ax.set_title(f"Augmentation pipeline comparison ({EPOCHS_AUG} epochs)")
ax.tick_params(axis="x", rotation=15)
plt.tight_layout()
plt.show()

Every pipeline trains the identical architecture for the identical
number of epochs; the only thing that differs is what the network is
shown each time it sees "the same" training image. Stacking more
transforms costs nothing at inference time (augmentation only ever
touches training data) and, within this small-scale experiment, the
richer pipelines are read directly off the validation-accuracy bars above
rather than assumed.

## Dropout and Weight Decay Ablation

4a proved that PyTorch's `weight_decay` optimiser argument, under **plain
SGD**, is algebraically identical to adding an $L_2$ penalty to the loss.
This grid exercises that guarantee directly: every combination of dropout
rate (`nn.Dropout`) and `weight_decay` is trained with plain
`optim.SGD` — no momentum, no adaptive scaling — so the equivalence 4a
derived holds exactly, with no adaptive-optimiser caveat to worry about.

In [ ]:
DROPOUT_RATES = [0.0, 0.3, 0.5]
WEIGHT_DECAYS = [0.0, 0.005, 0.02]
EPOCHS_DW = 60

dw_rows = []
dw_curves = {}
for dropout_prob in DROPOUT_RATES:
    for wd in WEIGHT_DECAYS:
        model = build_model(dropout_prob=dropout_prob).to(device)
        optimizer = optim.SGD(model.parameters(), lr=0.03, weight_decay=wd)
        train_losses, val_losses = [], []
        for epoch in range(EPOCHS_DW):
            train_loss = train_epoch(model, optimizer, None, epoch_seed=epoch)
            val_loss, val_acc = evaluate(model, X_val, y_val)
            train_losses.append(train_loss)
            val_losses.append(val_loss)
        dw_curves[(dropout_prob, wd)] = (train_losses, val_losses)
        dw_rows.append({"dropout": dropout_prob, "weight_decay": wd,
                         "final_train_loss": train_losses[-1], "val_loss": val_loss, "val_acc": val_acc})

dw_df = pd.DataFrame(dw_rows)
print(dw_df.to_string(index=False, float_format=lambda v: f"{v:.3f}"))

fig, ax = plt.subplots(figsize=(7, 5))
for dropout_prob in DROPOUT_RATES:
    _, val_losses = dw_curves[(dropout_prob, 0.0)]
    ax.plot(val_losses, label=f"dropout={dropout_prob}, weight_decay=0")
ax.set_xlabel("epoch")
ax.set_ylabel("validation loss")
ax.set_title("Learning curves: dropout rate alone")
ax.legend(fontsize=8)
ax.grid(alpha=0.3)
plt.show()

best_dw = dw_df.sort_values("val_acc", ascending=False).iloc[0]
print(f"\nbest config: dropout={best_dw['dropout']}, weight_decay={best_dw['weight_decay']}, "
      f"val accuracy={best_dw['val_acc']:.3f}")

Every row in this table was trained with exactly the same optimiser
class and the same number of epochs; only `dropout_prob` and
`weight_decay` change. The validation-loss curves make the mechanism
visible directly: heavier dropout keeps validation loss from turning
upward as early as the undropped run does, the same story 4a told with
hand-written masks, now produced by one keyword argument.

## Early Stopping in Practice

4a's early stopping looked back at a finished curve and picked the best
epoch after the fact. A real training loop cannot see the future: it must
decide, epoch by epoch, whether to keep going. The standard mechanism is
**patience** — track the best validation loss seen so far and a snapshot
of the weights that achieved it; stop once a fixed number of epochs have
passed with no improvement; restore the snapshot rather than keeping
whatever the final epoch happened to produce.

In [ ]:
def train_with_early_stopping(model, optimizer, transform=None, patience=10, max_epochs=150, min_delta=1e-4):
    best_val_loss = float("inf")
    best_state, best_epoch = None, -1
    epochs_without_improvement = 0
    history = {"train_loss": [], "val_loss": [], "val_acc": []}
    for epoch in range(max_epochs):
        train_loss = train_epoch(model, optimizer, transform, epoch_seed=epoch)
        val_loss, val_acc = evaluate(model, X_val, y_val)
        history["train_loss"].append(train_loss)
        history["val_loss"].append(val_loss)
        history["val_acc"].append(val_acc)
        if val_loss < best_val_loss - min_delta:
            best_val_loss, best_epoch = val_loss, epoch
            best_state = copy.deepcopy(model.state_dict())
            epochs_without_improvement = 0
        else:
            epochs_without_improvement += 1
            if epochs_without_improvement >= patience:
                break
    final_state = copy.deepcopy(model.state_dict())
    return history, best_state, best_epoch, final_state


es_model = build_model().to(device)
es_optimizer = optim.Adam(es_model.parameters(), lr=1e-3)
es_history, es_best_state, es_best_epoch, es_final_state = train_with_early_stopping(es_model, es_optimizer)
stopped_epoch = len(es_history["train_loss"]) - 1
print(f"stopped after epoch {stopped_epoch} (patience exhausted); best epoch was {es_best_epoch}")

fig, ax = plt.subplots(figsize=(7, 4.5))
ax.plot(es_history["train_loss"], "--", label="train loss")
ax.plot(es_history["val_loss"], label="val loss")
ax.axvline(es_best_epoch, color="C2", linestyle=":", label=f"best checkpoint (epoch {es_best_epoch})")
ax.axvline(stopped_epoch, color="C3", linestyle=":", label=f"stopped (epoch {stopped_epoch})")
ax.set_xlabel("epoch")
ax.set_ylabel("loss")
ax.set_title("Early stopping with patience and checkpoint restoration")
ax.legend(fontsize=8)
ax.grid(alpha=0.3)
plt.show()

es_model.load_state_dict(es_final_state)
_, test_acc_final = evaluate(es_model, X_test, y_test)
es_model.load_state_dict(es_best_state)
_, test_acc_best = evaluate(es_model, X_test, y_test)
print(f"test accuracy, final-epoch weights:      {test_acc_final:.3f}")
print(f"test accuracy, best-checkpoint weights:  {test_acc_best:.3f}")

Training keeps running for `patience` epochs past the best validation
loss before stopping — that lag is the price of not being able to see the
future, and it is also what separates real early stopping from 4a's
after-the-fact version. Restoring the checkpoint from the best epoch
rather than keeping the final weights is what actually captures the
benefit: the final epoch has had `patience` extra epochs to drift further
into overfitting, while the restored checkpoint has not.

## Learning Curves

Everything above varied one factor at a time. This closing experiment
combines the winning dropout/weight-decay configuration from the
ablation, the strongest augmentation pipeline, and early stopping into one
**fully regularised** model, and compares its learning curves directly
against an unregularised baseline trained the same way — the practical
counterpart of 4a's baseline-vs-technique comparisons, all four
techniques applied together instead of one at a time.

In [ ]:
best_pipeline_name = aug_df.iloc[0]["pipeline"]
best_transform = pipelines[best_pipeline_name]
print(f"using augmentation pipeline: {best_pipeline_name}")
print(f"using dropout={best_dw['dropout']}, weight_decay={best_dw['weight_decay']}")

baseline_model = build_model(dropout_prob=0.0).to(device)
baseline_optimizer = optim.Adam(baseline_model.parameters(), lr=1e-3)
baseline_history, baseline_best_state, baseline_best_epoch, _ = train_with_early_stopping(
    baseline_model, baseline_optimizer, transform=None)

reg_model = build_model(dropout_prob=float(best_dw["dropout"])).to(device)
reg_optimizer = optim.SGD(reg_model.parameters(), lr=0.03, weight_decay=float(best_dw["weight_decay"]))
reg_history, reg_best_state, reg_best_epoch, _ = train_with_early_stopping(
    reg_model, reg_optimizer, transform=best_transform)

fig, axes = plt.subplots(1, 2, figsize=(11, 4.5))
axes[0].plot(baseline_history["train_loss"], "--", alpha=0.6, color="C0")
axes[0].plot(baseline_history["val_loss"], color="C0", label="baseline (no regularisation)")
axes[0].plot(reg_history["train_loss"], "--", alpha=0.6, color="C1")
axes[0].plot(reg_history["val_loss"], color="C1", label="fully regularised")
axes[0].set_xlabel("epoch"); axes[0].set_ylabel("loss (solid=val, dashed=train)")
axes[0].set_title("Loss"); axes[0].legend(fontsize=8); axes[0].grid(alpha=0.3)

axes[1].plot(baseline_history["val_acc"], color="C0", label="baseline")
axes[1].plot(reg_history["val_acc"], color="C1", label="fully regularised")
axes[1].set_xlabel("epoch"); axes[1].set_ylabel("validation accuracy")
axes[1].set_title("Validation accuracy"); axes[1].legend(fontsize=8); axes[1].grid(alpha=0.3)
plt.tight_layout()
plt.show()

baseline_model.load_state_dict(baseline_best_state)
reg_model.load_state_dict(reg_best_state)
_, baseline_test_acc = evaluate(baseline_model, X_test, y_test)
_, reg_test_acc = evaluate(reg_model, X_test, y_test)
print(f"baseline (best checkpoint):         stopped epoch {len(baseline_history['train_loss'])-1}, "
      f"best epoch {baseline_best_epoch}, test accuracy {baseline_test_acc:.3f}")
print(f"fully regularised (best checkpoint): stopped epoch {len(reg_history['train_loss'])-1}, "
      f"best epoch {reg_best_epoch}, test accuracy {reg_test_acc:.3f}")

The baseline's train and validation losses separate quickly — the same
overfitting 4a demonstrated, now on a genuine held-out validation split.
The fully regularised model's two curves track each other far more
closely for far longer, which is the qualitative signature every
technique in this notebook aims at: keep the network learning genuine
structure for as long as possible before validation performance stops
tracking training performance.

## Key Takeaways

- Production practice reserves a third split, **validation**, for every
  decision made during development, and touches the **test** split
  exactly once — every experiment in this notebook (augmentation choice,
  dropout/weight-decay grid, the stopping point) used validation data,
  never test data, until the final comparison.
- Stacking `torchvision.transforms` (`RandomHorizontalFlip`, `RandomCrop`,
  `ColorJitter`) into an augmentation pipeline and comparing pipelines
  under otherwise identical training reproduces 4a's single hand-written
  augmentation as one point on a richer, measurable spectrum.
- A **dropout x weight-decay ablation trained entirely with plain SGD**
  exercises 4a's proven weight-decay/$L_2$ equivalence directly — the
  `weight_decay` keyword argument behaves exactly as derived, with no
  adaptive-optimiser caveat, because no adaptive optimiser is involved.
- **Early stopping as a real mechanism** — patience, a tracked best
  validation loss, and checkpoint restoration — differs from 4a's
  after-the-fact version in one crucial way: it does not know the future,
  so it always trains `patience` epochs past the best point before
  stopping, and restoring the best checkpoint (not the final weights) is
  what captures the benefit.
- Combining the winning augmentation, dropout, weight decay and early
  stopping into one model kept its train and validation curves close
  together for far longer than the unregularised baseline — the same
  qualitative signature 4a's individual techniques each produced alone,
  now compounding together.